# Cellpose Batch Image Segmentation

This notebook segments all images in a folder using Cellpose with specific configuration tuning parameters and stores the results in the configured output folder.

## Configuration
- Input root: `T:/260402SA11`
- Output root: `T:/260402SA11/Analysis/Segmentation_260504`
- GPU acceleration enabled
- Custom cellpose parameters: diameter=30, flow_threshold=3, cellprob_threshold=1.2, batch_size=32

In [3]:
import os
import subprocess
import json
from pathlib import Path

# Load configuration
config = {
    "network_drives": {
        "T:": "//bs-hpsvr16/TimelapseData"
    },
    "pipeline_tuning": {
        "fetch_workers": 2,
        "gpu_workers": 1,
        "max_staged_folders": 2
    },
    "input_root": "T:/260402SA11",
    "output_root": "T:/260402SA11/Analysis/Segmentation_260504",
    "local_scratch_root": "D:/pipeline_scratch_SA",
    "folder_filter": "*_p*",
    "cellpose": {
        "use_gpu": True,
        "img_filter": "w00",
        "diameter": 30.0,
        "flow_threshold": 3,
        "cellprob_threshold": 1.2,
        "batch_size": 32,
        "norm_percentile_low": 1.0,
        "norm_percentile_high": 99.0,
        "save_png": True,
        "no_npy": True,
        "verbose": True
    }
}

# Define paths from config
input_root = config["input_root"]
output_root = config["output_root"]
cellpose_config = config["cellpose"]

# Create output directory if it doesn't exist
os.makedirs(output_root, exist_ok=True)

print(f"Configuration loaded successfully")
print(f"Input root: {input_root}")
print(f"Output root: {output_root}")
print(f"Output directory created: {os.path.exists(output_root)}")
print(f"\nCellpose configuration:")
for key, value in cellpose_config.items():
    print(f"  {key}: {value}")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'T:/'

In [ ]:
# Run Cellpose segmentation on all images
# Build the cellpose command using configuration parameters

# Change to the cellpose directory first
cellpose_dir = os.path.dirname(os.path.abspath(__file__))
if not cellpose_dir or cellpose_dir == "<stdin>":
    # If running in Jupyter, use the workspace root
    cellpose_dir = r"C:/Users/agreicius/Desktop/cellpose-main"

os.chdir(cellpose_dir)
print(f"Changed to directory: {os.getcwd()}")

# Build the cellpose command with all parameters from config
cellpose_command = [
    "uv", "run", "cellpose",
    "--dir", input_root,
    "--outdir", output_root,
    "--diameter", str(cellpose_config["diameter"]),
    "--flow_threshold", str(cellpose_config["flow_threshold"]),
    "--cellprob_threshold", str(cellpose_config["cellprob_threshold"]),
    "--batch_size", str(cellpose_config["batch_size"]),
    "--norm_percentile_low", str(cellpose_config["norm_percentile_low"]),
    "--norm_percentile_high", str(cellpose_config["norm_percentile_high"]),
]

# Add optional flags
if cellpose_config["use_gpu"]:
    cellpose_command.append("--use_gpu")

if cellpose_config["save_png"]:
    cellpose_command.append("--save_png")

if cellpose_config["no_npy"]:
    cellpose_command.append("--no_npy")

if cellpose_config["verbose"]:
    cellpose_command.append("--verbose")

print("\nRunning Cellpose segmentation...")
print(f"Command: {' '.join(cellpose_command)}")
print(f"\nStarting segmentation with parameters:")
print(f"  Input folder: {input_root}")
print(f"  Output folder: {output_root}")
print(f"  Diameter: {cellpose_config['diameter']}")
print(f"  Flow threshold: {cellpose_config['flow_threshold']}")
print(f"  Cell probability threshold: {cellpose_config['cellprob_threshold']}")
print(f"  Batch size: {cellpose_config['batch_size']}")
print(f"  GPU enabled: {cellpose_config['use_gpu']}\n")

In [ ]:
try:
    result = subprocess.run(cellpose_command, check=True, capture_output=False, text=True)
    print("\nSegmentation completed successfully!")
except subprocess.CalledProcessError as e:
    print(f"\nError during segmentation: {e}")
    print(e.stderr)

In [ ]:
# Verify segmentation results
import glob

# List all segmented PNG files in the output folder (recursively)
segmented_files = glob.glob(os.path.join(output_root, "**", "*_seg.png"), recursive=True)

print(f"\nSegmentation Results:")
print(f"Total segmented images: {len(segmented_files)}")

if segmented_files:
    print("\nSegmented files:")
    for file in sorted(segmented_files)[:20]:  # Show first 20
        print(f"  - {file}")
    
    if len(segmented_files) > 20:
        print(f"  ... and {len(segmented_files) - 20} more files")
else:
    print("No segmented images found in output directory")

# Also list any generated flow and probability files
flow_files = glob.glob(os.path.join(output_root, "**", "*_flows.tif"), recursive=True)
print(f"\nFlow files generated: {len(flow_files)}")